In [0]:
# Caminho onde estão os arquivos CSV
input_path = "/Volumes/workspace/default/Inputs/"

# Conferir os arquivos disponíveis
display(dbutils.fs.ls(input_path))

# Criar o schema Bronze caso ainda não exista
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/Inputs/Chamados_Hora.CSV,Chamados_Hora.CSV,11547467,1792951133000
dbfs:/Volumes/workspace/default/Inputs/base_atendentes.csv,base_atendentes.csv,1525,1792951098000
dbfs:/Volumes/workspace/default/Inputs/base_motivos.csv,base_motivos.csv,800,1792951098000
dbfs:/Volumes/workspace/default/Inputs/canais.csv,canais.csv,219,1792951098000
dbfs:/Volumes/workspace/default/Inputs/chamados.csv,chamados.csv,14825050,1792951135000
dbfs:/Volumes/workspace/default/Inputs/clientes.csv,clientes.csv,4568046,1792951120000
dbfs:/Volumes/workspace/default/Inputs/custos.csv,custos.csv,3343365,1792951116000
dbfs:/Volumes/workspace/default/Inputs/pesquisa_satisfacao.csv,pesquisa_satisfacao.csv,1705822,1792951105000


DataFrame[]

In [0]:
from pyspark.sql import functions as F


def carregar_bronze(nome_arquivo, nome_tabela, encoding="UTF-8"):

    # 1. Montar o caminho completo do arquivo
    caminho = input_path + nome_arquivo

    print(f"Iniciando leitura: {nome_arquivo}")

    # 2. Ler o arquivo CSV
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("sep", ",")
        .option("encoding", encoding)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(caminho)
    )

    # 3. Padronizar os nomes das colunas
    df = df.toDF(
        *[coluna.strip().lower() for coluna in df.columns]
    )

    # 4. Validar se o arquivo possui colunas
    if len(df.columns) == 0:
        raise ValueError(
            f"O arquivo {nome_arquivo} não possui colunas."
        )

    # 5. Validar nomes duplicados após a padronização
    if len(df.columns) != len(set(df.columns)):
        raise ValueError(
            f"O arquivo {nome_arquivo} possui "
            "nomes de colunas duplicados."
        )

    # 6. Conferir a estrutura e a quantidade de registros
    df.printSchema()

    quantidade_origem = df.count()

    print(
        f"Registros lidos de {nome_arquivo}: "
        f"{quantidade_origem}"
    )

    # 7. Adicionar a data e hora da ingestão
    df = df.withColumn(
        "ingestion_datetime",
        F.current_timestamp()
    )

    # 8. Gravar na tabela Bronze
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(nome_tabela)
    )

    # 9. Ler a tabela gravada para validar o resultado
    df_bronze = spark.table(nome_tabela)

    quantidade_bronze = df_bronze.count()

    print(f"Tabela carregada: {nome_tabela}")
    print(f"Registros de origem: {quantidade_origem}")
    print(f"Registros na Bronze: {quantidade_bronze}")

    # 10. Verificar se as quantidades correspondem
    if quantidade_origem != quantidade_bronze:
        raise ValueError(
            f"A quantidade de registros não corresponde "
            f"na tabela {nome_tabela}."
        )

    print("Carga concluída com sucesso.")

    return df_bronze

In [0]:

# Arquivo de origem -> tabela Bronze -> codificação
arquivos_bronze = [
    ("base_atendentes.csv",
     "workspace.bronze.base_atendentes", "windows-1252"),

    ("base_motivos.csv",
     "workspace.bronze.base_motivos", "windows-1252"),

    ("canais.csv",
     "workspace.bronze.canais", "windows-1252"),

    ("chamados.csv",
     "workspace.bronze.chamados", "windows-1252"),

    ("Chamados_Hora.CSV",
     "workspace.bronze.chamados_hora", "windows-1252"),

    ("clientes.csv",
     "workspace.bronze.clientes", "windows-1252"),

    ("custos.csv",
     "workspace.bronze.custos", "windows-1252"),

    ("pesquisa_satisfacao.csv",
     "workspace.bronze.pesquisa_satisfacao", "windows-1252")
]

# Executar a carga de cada arquivo
for nome_arquivo, nome_tabela, encoding in arquivos_bronze:
    carregar_bronze(nome_arquivo, nome_tabela, encoding)

Iniciando leitura: base_atendentes.csv
root
 |-- id_atendente: string (nullable = true)
 |-- nome_atendente: string (nullable = true)
 |-- nivel_atendimento: string (nullable = true)
 |-- data_admissao: string (nullable = true)
 |-- data_desligamento: string (nullable = true)
 |-- status_atendente: string (nullable = true)

Registros lidos de base_atendentes.csv: 25
Tabela carregada: workspace.bronze.base_atendentes
Registros de origem: 25
Registros na Bronze: 25
Carga concluída com sucesso.
Iniciando leitura: base_motivos.csv
root
 |-- id_motivo: string (nullable = true)
 |-- nome_motivo: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- criticidade: string (nullable = true)

Registros lidos de base_motivos.csv: 14
Tabela carregada: workspace.bronze.base_motivos
Registros de origem: 14
Registros na Bronze: 14
Carga concluída com sucesso.
Iniciando leitura: canais.csv
root
 |-- id_canal: string (nullable = true)
 |-- nome_canal: string (nullable = true)
 |-- tipo_c

In [0]:

# Verificar as tabelas carregadas na camada Bronze
for _, nome_tabela, _ in arquivos_bronze:
    df_bronze = spark.table(nome_tabela)

    print(f"\nTabela: {nome_tabela}")
    print(f"Quantidade de registros: {df_bronze.count()}")

    display(df_bronze.limit(5))


Tabela: workspace.bronze.base_atendentes
Quantidade de registros: 25


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,ingestion_datetime
1,Marcela FranÃ§a,1,18/04/2019,16/11/2025,inativo,2026-10-25T20:57:58.028Z
2,Zilda Esteves,1,12/09/2023,24/01/2026,inativo,2026-10-25T20:57:58.028Z
3,William Marques,1,28/05/2019,15/04/2025,inativo,2026-10-25T20:57:58.028Z
4,Solange Silveira,NÃ­vel 1,07/10/2019,null,ativo,2026-10-25T20:57:58.028Z
5,Joaquim Barros,NÃ­vel 1,06/12/2020,23/05/2026,inativo,2026-10-25T20:57:58.028Z



Tabela: workspace.bronze.base_motivos
Quantidade de registros: 14


id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime
11,Bloqueio de cartÃ£o,Cartao,Alta,2026-10-25T20:58:02.106Z
7,Compra nÃ£o autorizada,Cartao,alta,2026-10-25T20:58:02.106Z
13,DÃºvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-25T20:58:02.106Z
14,TransferÃªncia de agÃªncia,null,Media,2026-10-25T20:58:02.106Z
9,Problema com aplicativo,Atendimento,Media,2026-10-25T20:58:02.106Z



Tabela: workspace.bronze.canais
Quantidade de registros: 4


...

id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura,ingestion_datetime
56324416706,Sérgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,14/10/2024,10,2026-10-25T20:58:19.717Z
97653229900,Leandro Magalhães Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10,2026-10-25T20:58:19.717Z
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,16/02/2019,25,2026-10-25T20:58:19.717Z
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,27/08/2025,20,2026-10-25T20:58:19.717Z
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,07/05/2026,15,2026-10-25T20:58:19.717Z



Tabela: workspace.bronze.custos
Quantidade de registros: 131069


id_custo,id_chamado,custo,ingestion_datetime
114049,110377,"0,39",2026-10-25T20:58:23.928Z
87679,83693,"R$ 1,46",2026-10-25T20:58:23.928Z
92429,88501,"12,24",2026-10-25T20:58:23.928Z
58033,53673,"24,43",2026-10-25T20:58:23.928Z
31491,26832,11.14reais,2026-10-25T20:58:23.928Z



Tabela: workspace.bronze.pesquisa_satisfacao
Quantidade de registros: 81021


id_pesquisa,id_chamado,nota_atendimento,ingestion_datetime
71619,115746,3,2026-10-25T20:58:28.468Z
13534,19389,4,2026-10-25T20:58:28.468Z
53852,83856,4,2026-10-25T20:58:28.468Z
5563,7771,5,2026-10-25T20:58:28.468Z
15959,22934,5,2026-10-25T20:58:28.468Z


In [0]:

# Verificar possíveis problemas de codificação nas tabelas Bronze

caracteres_suspeitos = ["�", "Ã§", "Ã£", "Ã¡", "Ã©", "Ã³", "Ãº", "Ã­", "Â"]

for _, nome_tabela, _ in arquivos_bronze:
    df = spark.table(nome_tabela)

    # Selecionar as colunas de texto
    colunas_texto = [
        campo.name
        for campo in df.schema.fields
        if campo.dataType.simpleString() == "string"
    ]

    # Procurar caracteres suspeitos em qualquer coluna de texto
    condicao = None

    for coluna in colunas_texto:
        for caractere in caracteres_suspeitos:
            teste = F.col(coluna).contains(caractere)
            condicao = teste if condicao is None else condicao | teste

    if condicao is not None:
        suspeitos = df.filter(condicao)
        quantidade = suspeitos.count()

        print(f"\nTabela: {nome_tabela}")
        print(f"Linhas com possíveis problemas: {quantidade}")

        if quantidade > 0:
            display(suspeitos.limit(5))
    else:
        print(f"\nTabela: {nome_tabela} - sem colunas de texto")



Tabela: workspace.bronze.base_atendentes
Linhas com possíveis problemas: 12


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,ingestion_datetime
1,Marcela FranÃ§a,1,18/04/2019,16/11/2025,inativo,2026-10-25T20:57:58.028Z
4,Solange Silveira,NÃ­vel 1,07/10/2019,null,ativo,2026-10-25T20:57:58.028Z
5,Joaquim Barros,NÃ­vel 1,06/12/2020,23/05/2026,inativo,2026-10-25T20:57:58.028Z
6,Roberta AssunÃ§Ã£o,1,29/10/2021,null,ativo,2026-10-25T20:57:58.028Z
7,Luiza GalvÃ£o,NÃ­vel 1,24/04/2022,null,ativo,2026-10-25T20:57:58.028Z



Tabela: workspace.bronze.base_motivos
Linhas com possíveis problemas: 8


id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime
11,Bloqueio de cartÃ£o,Cartao,Alta,2026-10-25T20:58:02.106Z
7,Compra nÃ£o autorizada,Cartao,alta,2026-10-25T20:58:02.106Z
13,DÃºvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-25T20:58:02.106Z
15,RenegociaÃ§Ã£o de dÃ­vida,Financeiro,alta,2026-10-25T20:58:02.106Z
4,"AlteraÃ§Ã£o de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,baixa,2026-10-25T20:58:02.106Z



Tabela: workspace.bronze.canais
Linhas com possíveis problemas: 0

Tabela: workspace.bronze.chamados
Linhas com possíveis problemas: 96917


id_chamado,id_cliente,motivo,canal,resolvido,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera,tempo_atendimento,id_atendente,ingestion_datetime
76696,74719850215,RenegociaÃ§Ã£o de dÃ­vida,atendimento inicial,Sim,null,null,null,49,441,4,2026-10-25T20:58:10.760Z
52060,36276971799,RenegociaÃ§Ã£o de dÃ­vida,Ura,NÃ£o,null,igual a hora de abertura,null,NULL,216,9,2026-10-25T20:58:10.760Z
25271,53061261807,RenegociaÃ§Ã£o de dÃ­vida,Atendimento_Inicial,N,null,null,null,96,384,2,2026-10-25T20:58:10.760Z
88817,28669204533,ContrataÃ§Ã£o de cartÃ£o adicional,Atendimento Inicial,NÃ£o,null,null,null,344,891,23,2026-10-25T20:58:10.760Z
89333,52402176464,RENEGOCIAÃ‡ÃƒO DE DÃ�VIDA,CHATBOT,N,null,igual a hora de abertura,null,NULL,104,3,2026-10-25T20:58:10.760Z



Tabela: workspace.bronze.chamados_hora
Linhas com possíveis problemas: 0

Tabela: workspace.bronze.clientes
Linhas com possíveis problemas: 0

Tabela: workspace.bronze.custos
Linhas com possíveis problemas: 0

Tabela: workspace.bronze.pesquisa_satisfacao
Linhas com possíveis problemas: 0
